Create a machine learning model to predict if a bank client is about to cancel the account.

In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder


In [2]:
df = pd.read_csv('./data/Churn.csv')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  str    
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  str    
 5   Gender           10000 non-null  str    
 6   Age              10000 non-null  int64  
 7   Tenure           9091 non-null   float64
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(3), int64(8), str(3)
memory usage: 1.1 MB


In [3]:
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2.0,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1.0,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8.0,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1.0,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2.0,125510.82,1,1,1,79084.10,0


The column 'Exited' shows if the costumer cancelled (1) or not (0). It is the model prediction objective.

## 1 - Cleaning

In [4]:
df.duplicated().sum()

np.int64(0)

In [5]:
df['CustomerId'].duplicated().sum()

np.int64(0)

There is no duplicated row or 'CustomerId'.

The columns 'RowNumber', 'CustomerId' and 'Surname' are not relevant for model training and can be discarded.

In [6]:
# df = df.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)
df = df.drop(['Surname'], axis=1)

There are null values in the 'Tenure' column. Verify if they are 0.

In [7]:
df['Tenure'].value_counts(dropna=False)

Tenure
1.0     952
2.0     950
8.0     933
3.0     928
5.0     927
7.0     925
NaN     909
4.0     885
9.0     882
6.0     881
10.0    446
0.0     382
Name: count, dtype: int64

These null are not 0, because there are observations with 0. Verify if there is something in common with these customers.

In [8]:
df[df['Tenure'].isnull()].sample(10, random_state=123)

,RowNumber,CustomerId,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
7811,7812,15687852,611,France,Male,30,NaN,104145.65,1,0,0,159629.64,0
3543,3544,15801218,675,France,Male,49,NaN,135133.39,1,0,1,179521.24,1
6471,6472,15800233,850,France,Female,40,NaN,0.00,2,1,0,35034.15,0
94,95,15676966,730,Spain,Male,42,NaN,0.00,2,0,1,85982.47,0
6187,6188,15751412,704,France,Male,36,NaN,114370.41,1,0,1,66810.48,0
7026,7027,15588123,677,France,Female,27,NaN,0.00,2,0,1,114685.92,0
5314,5315,15788068,743,Germany,Male,45,NaN,144677.19,3,1,0,22512.44,1
4445,4446,15736413,739,France,Male,29,NaN,0.00,2,1,1,164484.78,0
9006,9007,15756118,661,Spain,Male,20,NaN,0.00,1,1,0,110252.53,0
1960,1961,15623647,655,Spain,Female,36,NaN,135515.76,1,1,0,86013.96,0


There is nothing in common between them. These will be discarded.

In [9]:
df = df.dropna()

## 2 - Preparing

### 2.1 - Class Imbalance Analysis

In [10]:
df['Exited'].value_counts(normalize=True)

Exited
0    0.796062
1    0.203938
Name: proportion, dtype: float64

Only 20% of the values are 1. This will not be balanced now to compare the model before and after considering the imbalance.

### 2.2 - Encode Cathegorical Characteristics

In [11]:
df['Geography'].unique()

<StringArray>
['France', 'Spain', 'Germany']
Length: 3, dtype: str

In [12]:
df['Gender'].unique()

<StringArray>
['Female', 'Male']
Length: 2, dtype: str

Some models performs better with One-Hot Encoding, others with Label Encoding.

Two dfs will be created, one for each encoding technique.

#### 2.2.1 One-Hot Encoding

In [13]:
df_ohe = pd.get_dummies(df, drop_first=True)
df_ohe.head()

,RowNumber,CustomerId,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_Germany,Geography_Spain,Gender_Male
0,1,15634602,619,42,2.0,0.00,1,1,1,101348.88,1,False,False,False
1,2,15647311,608,41,1.0,83807.86,1,0,1,112542.58,0,False,True,False
2,3,15619304,502,42,8.0,159660.80,3,1,0,113931.57,1,False,False,False
3,4,15701354,699,39,1.0,0.00,2,0,0,93826.63,0,False,False,False
4,5,15737888,850,43,2.0,125510.82,1,1,1,79084.10,0,False,True,False


#### 2.2.2 Label Encoding

In [14]:
cat_columns = ['Geography', 'Gender']

encoder = OrdinalEncoder()
df_cat = pd.DataFrame(
    encoder.fit_transform(df[cat_columns]), 
    columns=cat_columns
)

df_le = df
df_le[cat_columns] = df_cat
df_le.head()


,RowNumber,CustomerId,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,619,0.0,0.0,42,2.0,0.00,1,1,1,101348.88,1
1,2,15647311,608,2.0,0.0,41,1.0,83807.86,1,0,1,112542.58,0
2,3,15619304,502,0.0,0.0,42,8.0,159660.80,3,1,0,113931.57,1
3,4,15701354,699,0.0,0.0,39,1.0,0.00,2,0,0,93826.63,0
4,5,15737888,850,2.0,0.0,43,2.0,125510.82,1,1,1,79084.10,0


## 3 - Model with Imbalance
Model without correcting the target imbalance.

### 3.1 - Splitting

In [ ]:
# split df_ohe into train (0.6), validation (0.2) and test (0.2)
df_train, df_val_test = train_test_split(df_ohe, test_size=0.4, random_state=123)
df_val, df_test = train_test_split(df_val_test, test_size=0.5, random_state=123)

features_train_ohe = df_train.drop('Exited', axis=1)
target_train_ohe = df_train['Exited']

features_val_ohe = df_val.drop('Exited', axis=1)
target_val_ohe = df_val['Exited']

features_test_ohe = df_test.drop('Exited', axis=1)
target_test_ohe = df_test['Exited']

In [ ]:
# split df_le into train (0.6), validation (0.2) and test (0.2)
df_train, df_val_test = train_test_split(df_le, test_size=0.4, random_state=123)
df_val, df_test = train_test_split(df_val_test, test_size=0.5, random_state=123)

features_train_le = df_train.drop('Exited', axis=1)
target_train_le = df_train['Exited']

features_val_le = df_val.drop('Exited', axis=1)
target_val_le = df_val['Exited']

features_test_le = df_test.drop('Exited', axis=1)
target_test_le = df_test['Exited']

### 3.2 LogisticRegression
Logistic Regression model performs better with One-hot encoding.

In [ ]:
model_lr = LogisticRegression(solver='liblinear', random_state=123)
model_lr.fit(features_train_ohe, target_train_ohe)

score_train = model_lr.score(features_train_ohe, target_train_ohe)
score_val = model_lr.score(features_val_ohe, target_val_ohe)

print(
    f'train score: {score_train:.3f}\n'
    f'val score: {score_val:.3f}'
)

train score: 0.796
val score: 0.808


### 3.2 - RandomForestClassifier

Random Forest model performs better with Label Encoding.

In [20]:
best_score_train = 0
best_score_val = 0
best_n_estimators = 0
best_depth = 0

for estimators in range(1, 21):
    # print(f'est: {estimators}')
    for depth in range(1, 21):
        # print(f'depth: {depth}')
        model = RandomForestClassifier(n_estimators=estimators,
                                       max_depth=depth,
                                       random_state=123)
        model.fit(features_train_le, target_train_le)

        score_train = model.score(features_train_le, target_train_le)
        score_val = model.score(features_val_le, target_val_le)

        if score_val > best_score_val:
            best_score_val = score_val
            best_score_train = score_train
            best_n_estimators = estimators
            best_depth = depth

model_rfc = RandomForestClassifier(n_estimators=best_n_estimators,
                               max_depth=best_depth,
                               random_state=123)
model_rfc.fit(features_train_le, target_train_le)

print(
    f'best_depth: {best_depth}\n'
    f'best_n_estimators: {best_n_estimators}\n'
    f'train score: {best_score_train:.3f}\n'
    f'val score: {best_score_val:.3f}'
)

best_depth: 13
best_n_estimators: 19
train score: 0.938
val score: 0.878


### 3.3 - DecisionTreeClassifier